 Q1:
 Colonnes nécessaires pour répondre aux deux objectifs

date : pour savoir si la transaction a eu lieu un dimanche ou un autre jour

produit : pour calculer le chiffre d’affaires (CA) par produit

prix_unitaire :pour calculer le montant de chaque ligne

quantite :pour calculer le montant total (prix × quantité)

remise_pct :pour ajuster le montant avec la remise

ville :pour tester la robustesse par ville

moyen_paiement :pour harmoniser les valeurs et corriger les incohérences

Q2: Création de l’indicateur is_dimanche et des sous-ensembles

In [1]:
import pandas as pd 
df =pd.read_csv('retail_synthetique_tp3.csv',parse_dates=['date'])
df['is_dimanche']=df['date'].dt.weekday==6
df_dim=df[df['is_dimanche']]
df_autres =df[~df['is_dimanche']]

Q3: Mon unité d’analyse est la ligne d’achat (chaque produit acheté dans une transaction).
Ce choix est pertinent car l’objectif est de comparer le chiffre d’affaires par produit entre le dimanche et les autres jours.
Si on prenait le ticket (transaction), on perdrait le détail du produit et on ne pourrait pas identifier les produits dominants le dimanche.

==> pretraitement<==

Q4:


In [2]:
doublons = df. duplicated()
print(doublons )

0       False
1       False
2       False
3       False
4       False
        ...  
3510     True
3511     True
3512     True
3513     True
3514     True
Length: 3515, dtype: bool


oui il se trouve les doublons

le nombre des doublons:

In [5]:
nb_doublons = df. duplicated().sum()
print("ND:" , nb_doublons )

ND: 15


je supremme les doublons mais reste la ligne qui doublies 

In [8]:
df = df.drop_duplicates().reset_index(drop=True)

    
nb_doublons = df. duplicated().sum()
print("le nombre de les doublons",nb_doublons )


le nombre de les doublons 0


Q5:les valeures manquantes:

In [10]:
nan =df.isnull().sum()
print(nan)

date                0
ville               0
magasin             0
transaction_id      0
client_id           0
sexe              110
age_client        117
categorie           0
produit             0
prix_unitaire       0
quantite            0
remise_pct        106
moyen_paiement      0
is_dimanche         0
dtype: int64


Les valeurs manquantes dans sexe et age_client n’impactent pas directement l’analyse du chiffre d’affaires ou la comparaison Dimanche/Autres jours.
En revanche, les valeurs manquantes dans remise_pct peuvent fausser légèrement le calcul du montant, mais peuvent être imputées par 0 (absence de remise).
Aucune valeur manquante n’est observée dans les variables critiques (prix_unitaire, quantite, date, produit)

Q6: 
 sexe / age_client : suppression

Les valeurs manquantes ont été remplacer age par mouyen age et sexe par inconu car ces variables ne sont pas essentielles pour répondre aux objectifs du client.

 remise_pct : imputation par 0

Lorsqu’il manque la remise, nous supposons qu’aucune remise n’a été appliquée (valeur = 0).

In [14]:
df['sexe']=df['sexe'].fillna('incnnu')

moy_age =df['age_client'].mean()
df['age_client']=df['age_client'].fillna(moy_age )

df['remise_pct']=df['remise_pct'].fillna(0)


Q7:harmonisation les libelles de moyen 

In [15]:
paiement_map = {
    'Carte bancaire': 'CB',
    'CB': 'CB',
    'cb': 'CB',
    'Espèces': 'Espèces',
    'Cash': 'Espèces',
    'cash': 'Espèces',
    'Chèque': 'Chèque',
    'cheque': 'Chèque',
    'Virement': 'Virement',
    'virement bancaire': 'Virement'
}


df['moyen_paiement'] = df['moyen_paiement'].replace(paiement_map)


Q8:CREETION MONTANT 

In [21]:
df['montant'] = df['prix_unitaire'] * (1 - df['remise_pct'] / 100)

Q9:

In [25]:
# Fonction pour résumé complet
def resume_stat(df, colonne='montant'):
    Q1 = df[colonne].quantile(0.25)
    Q3 = df[colonne].quantile(0.75)
    IQR = Q3 - Q1
    return pd.Series({
        'n': df[colonne].count(),
        'somme': df[colonne].sum(),
        'moyenne': df[colonne].mean(),
        'mediane': df[colonne].median(),
        'Q1': Q1,
        'Q3': Q3,
        'écart-type': df[colonne].std(),
        'IQR': IQR
    })

# Résumé pour Dimanche et Autres jours

resume_autres = resume_stat(df_autres)
resume_dimanche =resume_stat(df_dim)
print("Résumé Dimanche :\n", resume_dimanche)
print("\nRésumé Autres jours :\n", resume_autres)

KeyError: 'montant'

In [24]:
print(df.columns)


Index(['date', 'ville', 'magasin', 'transaction_id', 'client_id', 'sexe',
       'age_client', 'categorie', 'produit', 'prix_unitaire', 'quantite',
       'remise_pct', 'moyen_paiement', 'is_dimanche', 'montant_net',
       'montant'],
      dtype='object')
